# REST APIs — Authentication & Data Processing
## Working with API Keys and Transforming Responses into DataFrames
Real-world APIs require authentication. This notebook covers API keys, secure credential management, and transforming JSON responses into structured data with pandas.

In [1]:
import requests
import pandas as pd
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("OPENWEATHER_API_KEY")
print("API key loaded:", api_key is not None)

API key loaded: False


In [2]:
import requests
import pandas as pd
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("OPENWEATHER_API_KEY")
print("API key loaded:", api_key is not None)

API key loaded: True


In [3]:
def get_current_weather(city: str) -> dict:
    """Fetch current weather for a given city using OpenWeatherMap API."""
    base_url = "https://api.openweathermap.org/data/2.5/weather"
    
    params = {
        "q": city,
        "appid": api_key,
        "units": "metric"
    }
    
    response = requests.get(base_url, params=params)
    response.raise_for_status()
    
    return response.json()

data = get_current_weather("Barcelona")
print(data)

{'coord': {'lon': 2.159, 'lat': 41.3888}, 'weather': [{'id': 803, 'main': 'Clouds', 'description': 'broken clouds', 'icon': '04d'}], 'base': 'stations', 'main': {'temp': 23.49, 'feels_like': 23.38, 'temp_min': 23.11, 'temp_max': 23.97, 'pressure': 1012, 'humidity': 57, 'sea_level': 1012, 'grnd_level': 1004}, 'visibility': 10000, 'wind': {'speed': 3.13, 'deg': 153, 'gust': 4.92}, 'clouds': {'all': 78}, 'dt': 1780571709, 'sys': {'type': 2, 'id': 18549, 'country': 'ES', 'sunrise': 1780546761, 'sunset': 1780600817}, 'timezone': 7200, 'id': 3128760, 'name': 'Barcelona', 'cod': 200}


In [4]:
def parse_weather(data: dict) -> dict:
    """Extract relevant fields from OpenWeatherMap response."""
    return {
        "city": data["name"],
        "country": data["sys"]["country"],
        "temperature": data["main"]["temp"],
        "feels_like": data["main"]["feels_like"],
        "humidity": data["main"]["humidity"],
        "pressure": data["main"]["pressure"],
        "weather": data["weather"][0]["description"],
        "wind_speed": data["wind"]["speed"],
        "visibility": data["visibility"]
    }

parsed = parse_weather(data)
print(parsed)

{'city': 'Barcelona', 'country': 'ES', 'temperature': 23.49, 'feels_like': 23.38, 'humidity': 57, 'pressure': 1012, 'weather': 'broken clouds', 'wind_speed': 3.13, 'visibility': 10000}


In [5]:
cities = ["Barcelona", "Madrid", "Lisboa", "Paris", "Berlin", "Amsterdam"]

records = []

for city in cities:
    try:
        raw = get_current_weather(city)
        parsed = parse_weather(raw)
        records.append(parsed)
        print(f"✓ {city}")
    except requests.exceptions.HTTPError as e:
        print(f"✗ {city} — {e}")

df = pd.DataFrame(records)
df

✓ Barcelona
✓ Madrid
✓ Lisboa
✓ Paris
✓ Berlin
✓ Amsterdam


,city,country,temperature,feels_like,humidity,pressure,weather,wind_speed,visibility
0,Barcelona,ES,23.49,23.38,57,1012,broken clouds,3.13,10000
1,Madrid,ES,25.82,25.11,25,1011,few clouds,2.68,10000
2,Lisbon,PT,20.48,20.41,70,1019,few clouds,5.36,10000
3,Paris,FR,19.46,19.08,62,1005,overcast clouds,7.60,10000
4,Berlin,DE,23.20,23.17,61,1004,overcast clouds,4.92,10000
5,Amsterdam,NL,16.43,15.90,68,998,overcast clouds,6.26,4953


In [6]:
print("=== European Weather Summary ===\n")
print(f"Hottest city: {df.loc[df['temperature'].idxmax(), 'city']} ({df['temperature'].max()}°C)")
print(f"Coldest city: {df.loc[df['temperature'].idxmin(), 'city']} ({df['temperature'].min()}°C)")
print(f"Most humid: {df.loc[df['humidity'].idxmax(), 'city']} ({df['humidity'].max()}%)")
print(f"Strongest wind: {df.loc[df['wind_speed'].idxmax(), 'city']} ({df['wind_speed'].max()} m/s)")
print(f"\nAverage temperature: {df['temperature'].mean():.1f}°C")
print(f"Average humidity: {df['humidity'].mean():.1f}%")

=== European Weather Summary ===

Hottest city: Madrid (25.82°C)
Coldest city: Amsterdam (16.43°C)
Most humid: Lisbon (70%)
Strongest wind: Paris (7.6 m/s)

Average temperature: 21.5°C
Average humidity: 57.2%
